In [1]:
import os
os.environ["KERAS_BACKEND"] = "tensorflow"

import time
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import tensorflow as tf
import keras

from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

print("TensorFlow:", tf.__version__)
print("Keras:", keras.__version__)

TensorFlow: 2.21.0
Keras: 3.13.2


In [2]:
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

# Detecta automaticamente a raiz do projeto.
# Funciona se este notebook estiver na raiz, em 03_transfer_learning ou em 03_transfer_learning/deep_smote.
possible_roots = [
    Path("."),
    Path(".."),
    Path("../..")
]

PROJECT_ROOT = None

for root in possible_roots:
    if (root / "data" / "GroundTruth.csv").exists():
        PROJECT_ROOT = root.resolve()
        break

if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Não foi possível localizar a raiz do projeto. "
        "Verifique se existe data/GroundTruth.csv."
    )

DATA_DIR = PROJECT_ROOT / "data"
DEEPSMOTE_DIR = DATA_DIR / "deepsmote"
MODELS_DIR = PROJECT_ROOT / "models" / "deep_smote"

TRAIN_REAL_CSV = DEEPSMOTE_DIR / "train_real_split.csv"
TEST_REAL_CSV = DEEPSMOTE_DIR / "test_real_split.csv"
SYNTH_CSV = DEEPSMOTE_DIR / "metadata_deepsmote.csv"

MODEL_PATH = MODELS_DIR / "best_densenet169_deepsmote_saved_images.keras"

IMG_SIZE = (320, 320)
BATCH_SIZE = 16

classes = ["MEL", "NV", "BCC", "AKIEC", "BKL", "DF", "VASC"]
class_code = {label: idx for idx, label in enumerate(classes)}
class_name = {idx: label for label, idx in class_code.items()}

print("PROJECT_ROOT:", PROJECT_ROOT)
print("TRAIN_REAL_CSV:", TRAIN_REAL_CSV)
print("SYNTH_CSV:", SYNTH_CSV)
print("TEST_REAL_CSV:", TEST_REAL_CSV)
print("MODEL_PATH:", MODEL_PATH)
print("IMG_SIZE:", IMG_SIZE)
print("BATCH_SIZE:", BATCH_SIZE)

PROJECT_ROOT: /Users/alexandre/Documents/GitHub/skin-cancer-classification
TRAIN_REAL_CSV: /Users/alexandre/Documents/GitHub/skin-cancer-classification/data/deepsmote/train_real_split.csv
SYNTH_CSV: /Users/alexandre/Documents/GitHub/skin-cancer-classification/data/deepsmote/metadata_deepsmote.csv
TEST_REAL_CSV: /Users/alexandre/Documents/GitHub/skin-cancer-classification/data/deepsmote/test_real_split.csv
MODEL_PATH: /Users/alexandre/Documents/GitHub/skin-cancer-classification/models/best_densenet169_deepsmote_saved_images.keras
IMG_SIZE: (320, 320)
BATCH_SIZE: 16


In [3]:
required_files = [
    TRAIN_REAL_CSV,
    TEST_REAL_CSV,
    MODEL_PATH
]

for path in required_files:
    if not path.exists():
        raise FileNotFoundError(f"Arquivo não encontrado: {path}")

train_real_df = pd.read_csv(TRAIN_REAL_CSV)
test_df = pd.read_csv(TEST_REAL_CSV)

# Para avaliar o desempenho no conjunto usado no treinamento,
# o treino será composto por treino real + imagens sintéticas, se elas existirem.
if SYNTH_CSV.exists():
    synth_df = pd.read_csv(SYNTH_CSV)

    train_df = pd.concat(
        [train_real_df, synth_df],
        ignore_index=True
    )

    print("Usando treino aumentado: treino real + imagens sintéticas.")
    print("Treino real:", len(train_real_df))
    print("Sintéticas:", len(synth_df))
else:
    train_df = train_real_df.copy()

    print("metadata_deepsmote.csv não encontrado.")
    print("Usando apenas treino real.")

train_df = train_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

print("\nTotal treino avaliado:", len(train_df))
print("Total teste avaliado:", len(test_df))

print("\nDistribuição treino:")
print(train_df["class_name"].value_counts().reindex(classes))

print("\nDistribuição teste:")
print(test_df["class_name"].value_counts().reindex(classes))

Usando treino aumentado: treino real + imagens sintéticas.
Treino real: 8012
Sintéticas: 29536

Total treino avaliado: 37548
Total teste avaliado: 2003

Distribuição treino:
class_name
MEL      5364
NV       5364
BCC      5364
AKIEC    5364
BKL      5364
DF       5364
VASC     5364
Name: count, dtype: int64

Distribuição teste:
class_name
MEL       223
NV       1341
BCC       103
AKIEC      65
BKL       220
DF         23
VASC       28
Name: count, dtype: int64


In [4]:
def load_image_tf(image_path, label):
    image = tf.io.read_file(image_path)

    image = tf.image.decode_image(
        image,
        channels=3,
        expand_animations=False
    )

    image = tf.image.resize(image, IMG_SIZE)
    image = tf.cast(image, tf.float32)

    return image, label


def build_image_dataset(dataframe, batch_size=BATCH_SIZE, shuffle=False):
    image_paths = dataframe["image_path"].astype(str).values
    labels = dataframe["label"].astype("int32").values

    ds = tf.data.Dataset.from_tensor_slices((image_paths, labels))
    ds = ds.map(load_image_tf, num_parallel_calls=tf.data.AUTOTUNE)

    if shuffle:
        ds = ds.shuffle(
            buffer_size=min(len(dataframe), 4096),
            seed=SEED
        )

    ds = ds.batch(batch_size)
    ds = ds.prefetch(tf.data.AUTOTUNE)

    return ds


train_ds = build_image_dataset(
    train_df,
    batch_size=BATCH_SIZE,
    shuffle=False
)

test_ds = build_image_dataset(
    test_df,
    batch_size=BATCH_SIZE,
    shuffle=False
)

y_train = train_df["label"].astype("int32").values
y_test = test_df["label"].astype("int32").values

print("train_ds criado.")
print("test_ds criado.")
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)

train_ds criado.
test_ds criado.
y_train: (37548,)
y_test: (2003,)


## Carregamento do modelo salvo

In [5]:
# Correção de compatibilidade:
# Alguns modelos salvos com Keras/TensorFlow mais antigos podem conter argumentos
# antigos em BatchNormalization, como renorm, renorm_clipping e renorm_momentum.
# Este patch remove esses argumentos no momento do carregamento.

from keras.layers import BatchNormalization

_original_batchnorm_init = BatchNormalization.__init__

def _patched_batchnorm_init(self, *args, **kwargs):
    kwargs.pop("renorm", None)
    kwargs.pop("renorm_clipping", None)
    kwargs.pop("renorm_momentum", None)

    _original_batchnorm_init(self, *args, **kwargs)

BatchNormalization.__init__ = _patched_batchnorm_init

model = keras.models.load_model(
    MODEL_PATH,
    compile=False,
    safe_mode=False
)

model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()

Model: "densenet169_deepsmote_saved_images"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 320, 320, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ true_divide (TrueDivide)        │ (None, 320, 320, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ add (Add)                       │ (None, 320, 320, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ true_divide_1 (TrueDivide)      │ (None, 320, 320, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ densenet169 (Functional)        │ (None, 10, 10, 1664)   │    12,642,880 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 1664)           │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 512)            │       852,480 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 512)            │         2,048 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 512)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 256)            │       131,328 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 256)            │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 7)              │         1,799 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 13,631,559 (52.00 MB)

 Trainable params: 987,143 (3.77 MB)

 Non-trainable params: 12,644,416 (48.23 MB)

## Avaliação de loss e acurácia

In [6]:
start_time = time.time()

train_loss, train_accuracy = model.evaluate(
    train_ds,
    verbose=1
)

test_loss, test_accuracy = model.evaluate(
    test_ds,
    verbose=1
)

elapsed_time = time.time() - start_time

print("\nRESULTADOS DE LOSS E ACURÁCIA")
print(f"Train Loss: {train_loss:.4f}")
print(f"Train Accuracy: {train_accuracy:.4f}")
print(f"Test Loss: {test_loss:.4f}")
print(f"Test Accuracy: {test_accuracy:.4f}")
print(f"Tempo de avaliação: {elapsed_time:.2f} segundos")

 497/2347 ━━━━━━━━━━━━━━━━━━━━ 44:16 1s/step - accuracy: 0.9393 - loss: 0.2100

W0000 00:00:1782175236.119865 1025008 op_kernel.cc:1858] OP_REQUIRES failed at whole_file_read_ops.cc:117 : NOT_FOUND: ../../data/deepsmote/images/MEL/MEL_synth_000000.jpg; No such file or directory


 498/2347 ━━━━━━━━━━━━━━━━━━━━ 44:15 1s/step - accuracy: 0.9393 - loss: 0.2100

W0000 00:00:1782175237.568532 1025010 op_kernel.cc:1858] OP_REQUIRES failed at whole_file_read_ops.cc:117 : NOT_FOUND: ../../data/deepsmote/images/MEL/MEL_synth_000007.jpg; No such file or directory
W0000 00:00:1782175237.568562 1025013 op_kernel.cc:1858] OP_REQUIRES failed at whole_file_read_ops.cc:117 : NOT_FOUND: ../../data/deepsmote/images/MEL/MEL_synth_000011.jpg; No such file or directory
W0000 00:00:1782175237.568596 1025008 op_kernel.cc:1858] OP_REQUIRES failed at whole_file_read_ops.cc:117 : NOT_FOUND: ../../data/deepsmote/images/MEL/MEL_synth_000013.jpg; No such file or directory
W0000 00:00:1782175237.568634 1025013 op_kernel.cc:1858] OP_REQUIRES failed at whole_file_read_ops.cc:117 : NOT_FOUND: ../../data/deepsmote/images/MEL/MEL_synth_000012.jpg; No such file or directory
W0000 00:00:1782175237.568642 1025008 op_kernel.cc:1858] OP_REQUIRES failed at whole_file_read_ops.cc:117 : NOT_FOUND: ../../data/deepsmote/images/MEL/MEL_synth_000008.jpg; No such file or directory
W0000

 499/2347 ━━━━━━━━━━━━━━━━━━━━ 44:14 1s/step - accuracy: 0.9393 - loss: 0.2100

W0000 00:00:1782175239.130087 1025012 op_kernel.cc:1858] OP_REQUIRES failed at whole_file_read_ops.cc:117 : NOT_FOUND: ../../data/deepsmote/images/MEL/MEL_synth_000024.jpg; No such file or directory
W0000 00:00:1782175239.130105 1025009 op_kernel.cc:1858] OP_REQUIRES failed at whole_file_read_ops.cc:117 : NOT_FOUND: ../../data/deepsmote/images/MEL/MEL_synth_000025.jpg; No such file or directory
W0000 00:00:1782175239.130115 1025011 op_kernel.cc:1858] OP_REQUIRES failed at whole_file_read_ops.cc:117 : NOT_FOUND: ../../data/deepsmote/images/MEL/MEL_synth_000029.jpg; No such file or directory
W0000 00:00:1782175239.130131 1025006 op_kernel.cc:1858] OP_REQUIRES failed at whole_file_read_ops.cc:117 : NOT_FOUND: ../../data/deepsmote/images/MEL/MEL_synth_000027.jpg; No such file or directory
W0000 00:00:1782175239.130089 1025008 op_kernel.cc:1858] OP_REQUIRES failed at whole_file_read_ops.cc:117 : NOT_FOUND: ../../data/deepsmote/images/MEL/MEL_synth_000021.jpg; No such file or directory
W0000

 500/2347 ━━━━━━━━━━━━━━━━━━━━ 44:13 1s/step - accuracy: 0.9393 - loss: 0.2100

W0000 00:00:1782175240.745482 1025014 op_kernel.cc:1858] OP_REQUIRES failed at whole_file_read_ops.cc:117 : NOT_FOUND: ../../data/deepsmote/images/MEL/MEL_synth_000045.jpg; No such file or directory
W0000 00:00:1782175240.745479 1025009 op_kernel.cc:1858] OP_REQUIRES failed at whole_file_read_ops.cc:117 : NOT_FOUND: ../../data/deepsmote/images/MEL/MEL_synth_000043.jpg; No such file or directory
W0000 00:00:1782175240.745520 1025010 op_kernel.cc:1858] OP_REQUIRES failed at whole_file_read_ops.cc:117 : NOT_FOUND: ../../data/deepsmote/images/MEL/MEL_synth_000040.jpg; No such file or directory
W0000 00:00:1782175240.745479 1025013 op_kernel.cc:1858] OP_REQUIRES failed at whole_file_read_ops.cc:117 : NOT_FOUND: ../../data/deepsmote/images/MEL/MEL_synth_000038.jpg; No such file or directory
W0000 00:00:1782175240.745479 1025007 op_kernel.cc:1858] OP_REQUIRES failed at whole_file_read_ops.cc:117 : NOT_FOUND: ../../data/deepsmote/images/MEL/MEL_synth_000041.jpg; No such file or directory
W0000

NotFoundError: Graph execution error:

Detected at node ReadFile defined at (most recent call last):
<stack traces unavailable>
../../data/deepsmote/images/MEL/MEL_synth_000000.jpg; No such file or directory
	 [[{{node ReadFile}}]]
	 [[IteratorGetNext]] [Op:__inference_multi_step_on_iterator_19220]

## Predições

In [ ]:
start_time = time.time()

train_pred_probs = model.predict(
    train_ds,
    verbose=1
)

test_pred_probs = model.predict(
    test_ds,
    verbose=1
)

y_train_pred = np.argmax(train_pred_probs, axis=1)
y_test_pred = np.argmax(test_pred_probs, axis=1)

elapsed_time = time.time() - start_time

print("Predições geradas.")
print("train_pred_probs:", train_pred_probs.shape)
print("test_pred_probs:", test_pred_probs.shape)
print(f"Tempo de predição: {elapsed_time:.2f} segundos")

## Métricas gerais do treino

In [ ]:
print("Relatório de Classificação - TREINO:")
print(classification_report(
    y_train,
    y_train_pred,
    target_names=classes,
    digits=4,
    zero_division=0
))

train_acc = accuracy_score(y_train, y_train_pred)

train_macro_precision = precision_score(y_train, y_train_pred, average="macro", zero_division=0)
train_macro_recall = recall_score(y_train, y_train_pred, average="macro", zero_division=0)
train_macro_f1 = f1_score(y_train, y_train_pred, average="macro", zero_division=0)

train_weighted_precision = precision_score(y_train, y_train_pred, average="weighted", zero_division=0)
train_weighted_recall = recall_score(y_train, y_train_pred, average="weighted", zero_division=0)
train_weighted_f1 = f1_score(y_train, y_train_pred, average="weighted", zero_division=0)

print("\nMÉTRICAS GERAIS DO MODELO - TREINO")
print(f"Loss: {train_loss:.4f}")
print(f"Accuracy: {train_acc:.4f}")

print("\nMacro Average:")
print(f"  Precision: {train_macro_precision:.4f}")
print(f"  Recall: {train_macro_recall:.4f}")
print(f"  F1-score: {train_macro_f1:.4f}")

print("\nWeighted Average:")
print(f"  Precision: {train_weighted_precision:.4f}")
print(f"  Recall: {train_weighted_recall:.4f}")
print(f"  F1-score: {train_weighted_f1:.4f}")

train_metrics_df = pd.DataFrame([{
    "dataset": "Treino",
    "loss": train_loss,
    "accuracy": train_acc,
    "macro_precision": train_macro_precision,
    "macro_recall": train_macro_recall,
    "macro_f1": train_macro_f1,
    "weighted_precision": train_weighted_precision,
    "weighted_recall": train_weighted_recall,
    "weighted_f1": train_weighted_f1
}])

display(train_metrics_df)

## Métricas gerais do teste

In [ ]:
print("Relatório de Classificação - TESTE:")
print(classification_report(
    y_test,
    y_test_pred,
    target_names=classes,
    digits=4,
    zero_division=0
))

test_acc = accuracy_score(y_test, y_test_pred)

test_macro_precision = precision_score(y_test, y_test_pred, average="macro", zero_division=0)
test_macro_recall = recall_score(y_test, y_test_pred, average="macro", zero_division=0)
test_macro_f1 = f1_score(y_test, y_test_pred, average="macro", zero_division=0)

test_weighted_precision = precision_score(y_test, y_test_pred, average="weighted", zero_division=0)
test_weighted_recall = recall_score(y_test, y_test_pred, average="weighted", zero_division=0)
test_weighted_f1 = f1_score(y_test, y_test_pred, average="weighted", zero_division=0)

print("\nMÉTRICAS GERAIS DO MODELO - TESTE")
print(f"Loss: {test_loss:.4f}")
print(f"Accuracy: {test_acc:.4f}")

print("\nMacro Average:")
print(f"  Precision: {test_macro_precision:.4f}")
print(f"  Recall: {test_macro_recall:.4f}")
print(f"  F1-score: {test_macro_f1:.4f}")

print("\nWeighted Average:")
print(f"  Precision: {test_weighted_precision:.4f}")
print(f"  Recall: {test_weighted_recall:.4f}")
print(f"  F1-score: {test_weighted_f1:.4f}")

test_metrics_df = pd.DataFrame([{
    "dataset": "Teste",
    "loss": test_loss,
    "accuracy": test_acc,
    "macro_precision": test_macro_precision,
    "macro_recall": test_macro_recall,
    "macro_f1": test_macro_f1,
    "weighted_precision": test_weighted_precision,
    "weighted_recall": test_weighted_recall,
    "weighted_f1": test_weighted_f1
}])

display(test_metrics_df)

## Comparação treino vs teste

In [ ]:
metrics_comparison_df = pd.concat(
    [train_metrics_df, test_metrics_df],
    ignore_index=True
)

display(metrics_comparison_df)

plt.figure(figsize=(8, 5))
plt.bar(
    metrics_comparison_df["dataset"],
    metrics_comparison_df["accuracy"]
)
plt.title("Acurácia - Treino vs Teste")
plt.xlabel("Conjunto")
plt.ylabel("Acurácia")
plt.ylim(0, 1)

for i, value in enumerate(metrics_comparison_df["accuracy"]):
    plt.text(
        i,
        value,
        f"{value:.4f}",
        ha="center",
        va="bottom"
    )

plt.grid(True, alpha=0.3)
plt.show()

plt.figure(figsize=(8, 5))
plt.bar(
    metrics_comparison_df["dataset"],
    metrics_comparison_df["loss"]
)
plt.title("Perda - Treino vs Teste")
plt.xlabel("Conjunto")
plt.ylabel("Loss")

for i, value in enumerate(metrics_comparison_df["loss"]):
    plt.text(
        i,
        value,
        f"{value:.4f}",
        ha="center",
        va="bottom"
    )

plt.grid(True, alpha=0.3)
plt.show()

## Matriz de confusão percentual — treino

In [ ]:
cm_train = confusion_matrix(
    y_train,
    y_train_pred,
    labels=list(range(len(classes)))
)

cm_train_percent = cm_train.astype("float") / cm_train.sum(axis=1, keepdims=True) * 100
cm_train_percent = np.nan_to_num(cm_train_percent)

plt.figure(figsize=(8, 7))
im = plt.imshow(
    cm_train_percent,
    interpolation="nearest",
    cmap="Blues",
    vmin=0,
    vmax=100
)

plt.title("Matriz de Confusão (%) - Treino - DenseNet169 + DeepSMOTE")
plt.colorbar(im, label="%")

tick_marks = np.arange(len(classes))

plt.xticks(tick_marks, classes, rotation=45)
plt.yticks(tick_marks, classes)

for i in range(cm_train_percent.shape[0]):
    for j in range(cm_train_percent.shape[1]):
        color = "white" if cm_train_percent[i, j] > 50 else "black"

        plt.text(
            j,
            i,
            f"{cm_train_percent[i, j]:.1f}",
            ha="center",
            va="center",
            color=color,
            fontsize=8
        )

plt.ylabel("Classe Real")
plt.xlabel("Classe Predita")
plt.tight_layout()
plt.show()

cm_train_df = pd.DataFrame(
    cm_train_percent,
    index=classes,
    columns=classes
)

display(cm_train_df)

## Matriz de confusão percentual — teste

In [ ]:
cm_test = confusion_matrix(
    y_test,
    y_test_pred,
    labels=list(range(len(classes)))
)

cm_test_percent = cm_test.astype("float") / cm_test.sum(axis=1, keepdims=True) * 100
cm_test_percent = np.nan_to_num(cm_test_percent)

plt.figure(figsize=(8, 7))
im = plt.imshow(
    cm_test_percent,
    interpolation="nearest",
    cmap="Blues",
    vmin=0,
    vmax=100
)

plt.title("Matriz de Confusão (%) - Teste - DenseNet169 + DeepSMOTE")
plt.colorbar(im, label="%")

tick_marks = np.arange(len(classes))

plt.xticks(tick_marks, classes, rotation=45)
plt.yticks(tick_marks, classes)

for i in range(cm_test_percent.shape[0]):
    for j in range(cm_test_percent.shape[1]):
        color = "white" if cm_test_percent[i, j] > 50 else "black"

        plt.text(
            j,
            i,
            f"{cm_test_percent[i, j]:.1f}",
            ha="center",
            va="center",
            color=color,
            fontsize=8
        )

plt.ylabel("Classe Real")
plt.xlabel("Classe Predita")
plt.tight_layout()
plt.show()

cm_test_df = pd.DataFrame(
    cm_test_percent,
    index=classes,
    columns=classes
)

display(cm_test_df)

## Amostras de acertos e erros no teste

In [ ]:
test_eval_df = test_df.copy().reset_index(drop=True)

test_eval_df["y_true"] = y_test
test_eval_df["y_pred"] = y_test_pred
test_eval_df["correct"] = test_eval_df["y_true"] == test_eval_df["y_pred"]

correct_samples = test_eval_df[test_eval_df["correct"]].head(4)
incorrect_samples = test_eval_df[~test_eval_df["correct"]].head(4)

samples_to_plot = pd.concat(
    [correct_samples, incorrect_samples],
    axis=0
).reset_index(drop=True)

plt.figure(figsize=(12, 6))

for i, row in samples_to_plot.iterrows():
    image_path = row["image_path"]

    image = tf.io.read_file(image_path)
    image = tf.image.decode_image(
        image,
        channels=3,
        expand_animations=False
    )
    image = tf.image.resize(image, IMG_SIZE)
    image = image.numpy().astype("uint8")

    plt.subplot(2, 4, i + 1)
    plt.imshow(image)

    true_label = class_name[int(row["y_true"])]
    pred_label = class_name[int(row["y_pred"])]

    plt.title(f"{true_label} → {pred_label}", fontsize=10)
    plt.axis("off")

plt.tight_layout()
plt.show()